In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import shutil
import subprocess
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.linalg
import seaborn as sns
import torch

from sklearn.metrics import (
    adjusted_mutual_info_score as ami_score,
    adjusted_rand_score as ari_score,
    completeness_score as com_score,
    homogeneity_score as hom_score,
    normalized_mutual_info_score as nmi_score,
)
from sklearn.metrics.pairwise import euclidean_distances

PROJECT_ROOT = Path("/workspace/DVCAlign")
DATA_ROOT = PROJECT_ROOT / "DVCAlign" / "Data"

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import DVCAlign

RANDOM_SEED = 666
np.random.seed(RANDOM_SEED)

gpu_id = 2
if torch.cuda.is_available():
    if torch.cuda.device_count() > gpu_id:
        DEVICE = torch.device(f"cuda:{gpu_id}")
    else:
        DEVICE = torch.device("cuda:0")
else:
    DEVICE = torch.device("cpu")

SECTION_IDS = ["151673", "151674","151675", "151676"]

EMBEDDING_KEY = "DVCAlign"
LABEL_KEY = "Ground Truth"
SLICE_KEY = "slice_name"
BATCH_KEY = "batch_name"
CLUSTER_KEY = "mclust"
DOMAIN_KEY = "dvca_domain"

N_CLUSTERS = 7
SPATIAL_RADIUS = 200
N_TOP_GENES = 5000

sns.set(style="white", font_scale=1.1)
plt.rcParams["figure.dpi"] = 160
plt.rcParams["savefig.dpi"] = 300

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Device:", DEVICE)
print("Sections:", SECTION_IDS)

Project root: /workspace/DVCAlign
Data root: /workspace/DVCAlign/DVCAlign/Data
Device: cuda:2
Sections: ['151673', '151674', '151675', '151676']


In [2]:
def prepare_dvca_slice(
    section_id,
    data_root=DATA_ROOT,
    rad_cutoff=SPATIAL_RADIUS,
    n_top_genes=N_TOP_GENES
):
    input_dir = data_root / section_id

    adata = sc.read_visium(
        path=input_dir,
        count_file=f"{section_id}_filtered_feature_bc_matrix.h5",
        load_images=True,
    )

    adata.var_names_make_unique(join="++")

    ann_df = pd.read_csv(
        input_dir / f"{section_id}_truth.txt",
        sep="\t",
        header=None,
        index_col=0,
    )

    ann_df.columns = [LABEL_KEY]
    ann_df[ann_df.isna()] = "unknown"

    adata.obs[LABEL_KEY] = ann_df.loc[
        adata.obs_names,
        LABEL_KEY
    ].astype("category")

    adata.obs_names = [
        f"{name}_{section_id}"
        for name in adata.obs_names
    ]

    DVCAlign.Cal_Spatial_Net(
        adata,
        rad_cutoff=rad_cutoff
    )

    sc.pp.highly_variable_genes(
        adata,
        flavor="seurat_v3",
        n_top_genes=n_top_genes,
    )

    sc.pp.normalize_total(
        adata,
        target_sum=1e4
    )

    sc.pp.log1p(adata)

    adata = adata[
        :,
        adata.var["highly_variable"]
    ].copy()

    return adata


slice_adatas = []

for section_id in SECTION_IDS:
    print(f"Loading section {section_id}")
    slice_adatas.append(
        prepare_dvca_slice(section_id)
    )

adata_concat = ad.concat(
    slice_adatas,
    label=SLICE_KEY,
    keys=SECTION_IDS,
)

adata_concat.obs[LABEL_KEY] = (
    adata_concat.obs[LABEL_KEY]
    .astype("category")
)

adata_concat.obs[BATCH_KEY] = (
    adata_concat.obs[SLICE_KEY]
    .astype("category")
)

adj_concat = np.asarray(
    slice_adatas[0].uns["adj"].todense()
)

for batch_id in range(1, len(SECTION_IDS)):
    adj_concat = scipy.linalg.block_diag(
        adj_concat,
        np.asarray(
            slice_adatas[batch_id]
            .uns["adj"]
            .todense()
        )
    )

adata_concat.uns["edgeList"] = np.nonzero(adj_concat)

print("adata_concat.shape:", adata_concat.shape)

Loading section 151673


Variable names are not unique. To make them unique, call `.var_names_make_unique`.
Variable names are not unique. To make them unique, call `.var_names_make_unique`.


------Calculating spatial graph...
The graph contains 21124 edges, 3639 cells.
5.8049 neighbors per cell on average.
Loading section 151674


Variable names are not unique. To make them unique, call `.var_names_make_unique`.
Variable names are not unique. To make them unique, call `.var_names_make_unique`.


------Calculating spatial graph...
The graph contains 21258 edges, 3673 cells.
5.7876 neighbors per cell on average.
Loading section 151675


Variable names are not unique. To make them unique, call `.var_names_make_unique`.
Variable names are not unique. To make them unique, call `.var_names_make_unique`.


------Calculating spatial graph...
The graph contains 20762 edges, 3592 cells.
5.7801 neighbors per cell on average.
Loading section 151676


Variable names are not unique. To make them unique, call `.var_names_make_unique`.
Variable names are not unique. To make them unique, call `.var_names_make_unique`.


------Calculating spatial graph...
The graph contains 20052 edges, 3460 cells.
5.7954 neighbors per cell on average.
adata_concat.shape: (14364, 1125)


In [3]:
fit_config = dict(
    hidden_dims=[512, 32],
    n_epochs=1000,
    margin=1.0,
    knn_neigh=20,
    verbose=True,
    random_seed=666,
    device=DEVICE,
)

adata_concat = DVCAlign.train_DVCAlign(
    adata_concat,
    **fit_config
)

print("Training finished.")

DVCAlignModel(
  (conv1): GATConv(1125, 512, heads=1)
  (conv2): GATConv(512, 32, heads=1)
  (conv3): GATConv(32, 512, heads=1)
  (conv4): GATConv(512, 1125, heads=1)
  (residual): Linear(in_features=1125, out_features=32, bias=True)
  (view_gate): Linear(in_features=64, out_features=1, bias=True)
)
Use dual-view RC loss: lam_re=1.0, lam_rc=1.0, lam_dec=0.05, triplet_warmup_epochs=200, triplet_weight_max=0.7
Use spatial graph + local-expression graph dual views: aux_candidate_k=20, aux_expr_k=10, aux_pca_dim=30, spatial_fusion_weight=0.5
Triplet positive strategy: pick the closest spot among MNN candidates.
Use adaptive dual-view fusion: True; use confidence-aware triplet: True; use chain consistency: True
Pretrain DVCAlign encoder...


100%|██████████| 500/500 [01:20<00:00,  6.25it/s]


Train DVCAlign...


  0%|          | 0/500 [00:00<?, ?it/s]

Update spot triplets at epoch 500


...

Update spot triplets at epoch 800


 80%|████████  | 400/500 [01:28<00:19,  5.12it/s]

Update spot triplets at epoch 900


100%|██████████| 500/500 [01:50<00:00,  4.53it/s]

Training finished.


In [4]:
import shutil
import subprocess
import os

r_bin = shutil.which("R")

if r_bin is None:
    raise RuntimeError("当前环境找不到 R")

r_home = subprocess.check_output(
    [r_bin, "RHOME"],
    text=True
).strip()

os.environ["R_HOME"] = r_home

print("R executable:", r_bin)
print("R_HOME:", r_home)


adata_concat = DVCAlign.mclust_R(
    adata_concat,
    num_cluster=N_CLUSTERS,
    used_obsm=EMBEDDING_KEY,
)

adata_concat.obs[DOMAIN_KEY] = (
    adata_concat.obs[CLUSTER_KEY]
    .astype(str)
    .astype("category")
)

adata_eval = adata_concat[
    adata_concat.obs[LABEL_KEY].astype(str) != "unknown"
].copy()

ari_value = ari_score(
    adata_eval.obs[LABEL_KEY],
    adata_eval.obs[DOMAIN_KEY],
)

print(f"ARI = {ari_value:.3f}")

R executable: /opt/conda/envs/dvcalign/bin/R
R_HOME: /opt/conda/envs/dvcalign/lib/R


R[write to console]:                    __           __ 
   ____ ___  _____/ /_  _______/ /_
  / __ `__ \/ ___/ / / / / ___/ __/
 / / / / / / /__/ / /_/ (__  ) /_  
/_/ /_/ /_/\___/_/\__,_/____/\__/   version 6.0.1
Type 'citation("mclust")' for citing this R package in publications.



fitting ...
  |======================================================================| 100%
ARI = 0.636
